BRONZE LAYER

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("store_num", StringType(), True),
    StructField("store_name", StringType(), True),
    StructField("address", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip_code", DoubleType(), True),
    StructField("manager_info", StringType(), True),
    StructField("contact_info", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

df=spark.read.format("csv").schema(schema).option("header","true").option("quote",'"').option("escape",'"').option("mode","PERMISSIVE").option("columnNameOfCorruptRecord", "_corrupt_record").load("/Volumes/workspace/training/store_records/store_100_records.csv")

df.filter("_corrupt_record is not null").show()

df.write.format("delta").mode("overwrite").saveAsTable("workspace.training.store_100_records_bronze")

df_bronze=spark.read.table("workspace.training.store_100_records_bronze")
df_bronze.display()

SILVER LAYER

In [0]:
from pyspark.sql.functions import regexp_replace, expr, col, from_json, length, when
from pyspark.sql.types import StructType, StructField, StringType

manager_schema=StructType([
    StructField("manager_id", StringType(), True),
    StructField("manager_name", StringType(), True)
])

contact_schema=StructType([
    StructField("phone", StringType(), True),
    StructField("email", StringType(), True)
])

df_silver=spark.table("workspace.training.store_100_records_bronze")

df_silver = (df_silver.withColumn("zip_code", expr("regexp_replace(zip_code, '[^0-9]', '')"))).withColumn("zip_code",col("zip_code").cast("bigint"))

df_silver=df_silver.dropDuplicates().dropna(subset=['store_num']).drop("_corrupt_record")

df_silver=df_silver.withColumn("manager_info",from_json(col("manager_info"), manager_schema)).withColumn("contact_info",from_json(col("contact_info"), contact_schema))

df_silver=df_silver.withColumn("manager_id",col("manager_info.manager_id")).withColumn("manager_name",col("manager_info.manager_name")).withColumn("phone",col("contact_info.phone")).withColumn("email",col("contact_info.email")).drop("manager_info","contact_info")

df_silver=df_silver.withColumn("phone", expr("regexp_replace(phone, '[^0-9]', '')")).withColumn("phone", col("phone").try_cast("bigint"))

df_silver=df_silver.withColumn("phone_validity", when(length(col("phone"))!=10,"INVALID_PHONE").otherwise("Valid_phone"))
df_silver.display()

spark.sql("DROP TABLE IF EXISTS workspace.training.store_100_records_silver")

df_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.training.store_100_records_silver")